In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from xgboost import XGBClassifier

train_df = pd.read_pickle("../data/train.pkl")

In [2]:
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [3]:
train_df = train_df.sort_values(["Transaction_Day" , "Transaction_Time_Of_Day"]).reset_index(drop=True)

In [4]:
# Initializing PIPELINE

In [5]:
# NUMERICAL

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' , add_indicator=True)) ,
    ('scaler' , StandardScaler())
])

In [6]:
# CATEGORICAL

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant' , fill_value="Missing")) ,
    ('encoder' , OneHotEncoder(handle_unknown='ignore'))
])

In [7]:
x = train_df.drop(columns=["isFraud" , "TransactionID" ,"TransactionDT"])
y = train_df["isFraud"]

In [8]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5 , gap=0)

In [ ]:
oof_predictions = np.full(len(x) , np.nan)

for fold , (train_idx , val_idx) in enumerate(tscv.split(x) , start=1):
    x_train = x.iloc[train_idx].copy()
    x_val = x.iloc[val_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()

    numeric_cols = x_train.select_dtypes(include=["int64" , "float64" , "uint16" , "uint8"]).columns
    categorical_cols = x_train.select_dtypes(include=["object"]).columns

    # Combined Pipeline
    preprocessor = ColumnTransformer([
        ('num' , numeric_pipeline , numeric_cols) ,
        ('cat' , categorical_pipeline , categorical_cols)
    ])

    xgb_pipeline = Pipeline([
        ('preprocessor' , preprocessor) ,
        ('model' , XGBClassifier(random_state=42 , n_jobs=-1 , max_depth=10 , n_estimators=100))
    ])
    
    xgb_pipeline.fit(x_train , y_train)
    val_pred = xgb_pipeline.predict(x_val)
    val_prob = xgb_pipeline.predict_proba(x_val)[:,1]
    oof_predictions[val_idx] = val_prob

    roc_auc = roc_auc_score(y_val , val_prob)
    pr_auc = average_precision_score(y_val , val_prob)
    recall = recall_score(y_val , val_pred)
    f1 = f1_score(y_val , val_pred)
    precision = precision_score(y_val , val_pred)


    print(
        f"Fold {fold} | "
        f"ROC-AUC : {roc_auc:.4f} | " 
        f"PR-AUC : {pr_auc:.4f} | "
        f"Recall = {recall:.4f} | "
        f"F1 Score = {f1:.4f} | "
        f"Precision = {precision:.4f}")
    print(confusion_matrix(y_val , val_pred))

oof_mask = ~np.isnan(oof_predictions)
overall_roc_auc = roc_auc_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
overall_pr_auc = average_precision_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
print(f"OOF ROC-AUC : {overall_roc_auc:.4f}")
print(f"OOF PR-AUC : {overall_pr_auc:.4f}")

Fold 1 | ROC-AUC : 0.8614 | PR-AUC : 0.4862 | Recall = 0.2994 | F1 Score = 0.4378 | Precision = 0.8141
[[94890   226]
 [ 2317   990]]
Fold 2 | ROC-AUC : 0.8919 | PR-AUC : 0.5589 | Recall = 0.4167 | F1 Score = 0.5337 | Precision = 0.7420
[[93777   588]
 [ 2367  1691]]
Fold 3 | ROC-AUC : 0.8880 | PR-AUC : 0.5240 | Recall = 0.3304 | F1 Score = 0.4707 | Precision = 0.8180
[[94466   271]
 [ 2468  1218]]
Fold 4 | ROC-AUC : 0.9117 | PR-AUC : 0.5558 | Recall = 0.4121 | F1 Score = 0.5337 | Precision = 0.7572
[[94250   487]
 [ 2167  1519]]
Fold 5 | ROC-AUC : 0.9055 | PR-AUC : 0.5145 | Recall = 0.3865 | F1 Score = 0.5063 | Precision = 0.7335
[[94549   477]
 [ 2084  1313]]
OOF ROC-AUC : 0.8824
OOF PR-AUC : 0.5224


In [ ]:
xgboost_oof = pd.DataFrame({
    "isFraud" : y.values ,
    "xgboost_oof" : oof_predictions
})

xgboost_oof.to_csv("../outputs/oof/xgboost_oof.csv" , index=False)